# 챕터 1 — 한국 수출입의 20년 지형도 (2007–2026)

교역국과 품목 구성이 시간에 따라 **어떻게 변했는지**를 인과 해석 없이 서술하는 **기술통계(descriptive)** 노트북이다. 순위·비중·집중도·분포 같은 관측된 구조와 그 변화만 제시한다. 함께 있는 문서 `chapter01.md`가 이 결과들을 이야기로 엮은 것이다.

## 이 데이터베이스에 대하여 (출처·집계 기준)

이 데이터베이스는 관세청이 OpenAPI(품목별 국가별 수출입실적(GW), https://www.data.go.kr/data/15100475/openapi.do)로 공개하는 월별 수출입 통계를 **2007년 1월부터 2026년 3월까지** 한데 모아 하나의 파일로 만든 것이다.

**수치의 집계 기준(관세청 정의).** 수출입 신고 통관 자료를 국가 및 HS Code(2·4·6·10단위)별로 집계한 국가별 품목별 무역통계다. 금액은 미화(USD)이며, 수출은 FOB(신고금액), 수입은 CIF(과세가격) 기준이다. 중량은 순중량(kg). 국가는 수출은 최종목적국, 수입은 원산국을 원칙으로 하며 무역통계부호상 ISO 코드로 분류한다. 단순 통과물품이나 일시 반입·반출 물품은 제외된다(물적 자원의 증감이 없으므로). 통계는 매월 수출입 신고의 정정·취하를 반영해 전월까지 자료를 현행화한다(주기 1개월).

## 0. 규칙과 함정

- **금액**은 미화 **달러(USD)** — 수출 FOB·수입 CIF. **중량**은 순중량(kg). 단가는 `금액/중량`(USD/kg).
- **2026년 제외**(부분년). 연도 비교는 최신 완전연도 **2025**.
- **음수·0 중량 제외**(단가 계산 시).
- **HS 개정**은 `dim_hs6_concordance`로 연결(7절).
- 편의상 금액은 **억 달러**(1억 달러=100M USD)로 표기하는 표가 있다.

In [ ]:
import os
import duckdb
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
from matplotlib.ticker import MaxNLocator  # 연도축을 정수 눈금으로

DB_PATH = os.path.join("data", "processed", "kcsdb.duckdb")
if not os.path.exists(DB_PATH):
    raise FileNotFoundError(
        f"DB 없음: {DB_PATH}\n"
        "Releases에서 kcsdb.duckdb.gz를 받아 압축 해제 후 data/processed/ 에 놓으세요."
    )
con = duckdb.connect(DB_PATH, read_only=True)
def q(sql):
    return con.sql(sql).df()
print("연결 완료 —", f"{con.sql('SELECT COUNT(*) FROM fact_trade').fetchone()[0]:,}", "거래행")

## 3. 한눈에 보는 20년 — 연간 수출·수입·수지

In [ ]:
annual = q('''
    SELECT yyyymm//100 AS yr,
           ROUND(SUM(exp_dlr)/1e8,0) AS exp_eok,
           ROUND(SUM(imp_dlr)/1e8,0) AS imp_eok,
           ROUND(SUM(exp_dlr-imp_dlr)/1e8,0) AS bal_eok
    FROM fact_trade WHERE yyyymm//100 < 2026
    GROUP BY 1 ORDER BY 1
''')
print(annual.to_string(index=False))

plt.figure(figsize=(8,3.6))
plt.plot(annual['yr'], annual['exp_eok']/10, marker='o', label='Exports')
plt.plot(annual['yr'], annual['imp_eok']/10, marker='s', label='Imports')
plt.ylabel('Billion USD'); plt.xlabel('Year')
plt.title('Korea Annual Trade (2007-2025)')
plt.gca().xaxis.set_major_locator(MaxNLocator(integer=True))
plt.legend(); plt.grid(alpha=.3); plt.tight_layout(); plt.show()

## 4. 어디에 파는가 — 교역국 지형

In [ ]:
# 4.1 2025년 수출 상위 10개국 (억달러, 점유율%)
top_exp = q('''
    WITH t AS (SELECT SUM(exp_dlr) tot FROM fact_trade WHERE yyyymm//100=2025)
    SELECT COALESCE(c.name_ko_mofa,c.name_ko_kcs) AS 국가,
           ROUND(SUM(f.exp_dlr)/1e8,0) AS 수출_억달러,
           ROUND(100.0*SUM(f.exp_dlr)/(SELECT tot FROM t),1) AS 점유율
    FROM fact_trade f JOIN dim_country c USING(stat_cd)
    WHERE f.yyyymm//100=2025 GROUP BY 1 ORDER BY 수출_억달러 DESC LIMIT 10
''')
print(top_exp.to_string(index=False))

chart = q('''SELECT c.name_en AS en, ROUND(SUM(f.exp_dlr)/1e9,1) AS bil
             FROM fact_trade f JOIN dim_country c USING(stat_cd)
             WHERE f.yyyymm//100=2025 GROUP BY 1 ORDER BY bil DESC LIMIT 10''')
plt.figure(figsize=(8,4))
plt.barh(chart['en'][::-1], chart['bil'][::-1], color='steelblue')
plt.xlabel('Billion USD'); plt.title('Top 10 Export Partners (2025)')
plt.tight_layout(); plt.show()

In [ ]:
# 4.2 2025년 수입 상위 10개국
q('''
    WITH t AS (SELECT SUM(imp_dlr) tot FROM fact_trade WHERE yyyymm//100=2025)
    SELECT COALESCE(c.name_ko_mofa,c.name_ko_kcs) AS 국가,
           ROUND(SUM(f.imp_dlr)/1e8,0) AS 수입_억달러,
           ROUND(100.0*SUM(f.imp_dlr)/(SELECT tot FROM t),1) AS 점유율
    FROM fact_trade f JOIN dim_country c USING(stat_cd)
    WHERE f.yyyymm//100=2025 GROUP BY 1 ORDER BY 수입_억달러 DESC LIMIT 10
''')

In [ ]:
# 4.3 2007 vs 2025 수출 순위 비교
# 두 해 중 한 번이라도 상위 10위에 든 나라(합집합)를 잡고, 각국의 실제 순위를 양쪽 다 표기
q('''
    WITH y AS (
      SELECT f.yyyymm//100 yr, COALESCE(c.name_ko_mofa,c.name_ko_kcs) 국가, SUM(f.exp_dlr) v
      FROM fact_trade f JOIN dim_country c USING(stat_cd)
      WHERE f.yyyymm//100 IN (2007,2025) GROUP BY 1,2),
    r AS (SELECT yr, 국가, RANK() OVER (PARTITION BY yr ORDER BY v DESC) rk, v FROM y),
    top AS (SELECT DISTINCT 국가 FROM r WHERE rk<=10)   -- 어느 해든 상위 10위
    SELECT r.국가,
      MAX(CASE WHEN yr=2007 THEN rk END) AS 순위_2007,
      MAX(CASE WHEN yr=2025 THEN rk END) AS 순위_2025,
      ROUND(MAX(CASE WHEN yr=2007 THEN v END)/1e8,0) AS v2007_억,
      ROUND(MAX(CASE WHEN yr=2025 THEN v END)/1e8,0) AS v2025_억
    FROM r WHERE r.국가 IN (SELECT 국가 FROM top) GROUP BY r.국가
    ORDER BY 순위_2025 NULLS LAST, 순위_2007 NULLS LAST
''')

In [ ]:
# 4.4 2025년 수출 대륙별 비중 (외교부 대륙 구분; 상위 5개 = 99.9%)
q('''
    WITH t AS (SELECT SUM(exp_dlr) tot FROM fact_trade WHERE yyyymm//100=2025)
    SELECT COALESCE(c.continent_mofa,'(미분류)') AS 대륙,
           ROUND(SUM(f.exp_dlr)/1e8,0) AS 수출_억달러,
           ROUND(100.0*SUM(f.exp_dlr)/(SELECT tot FROM t),1) AS 점유율
    FROM fact_trade f LEFT JOIN dim_country c USING(stat_cd)
    WHERE f.yyyymm//100=2025 GROUP BY 1 ORDER BY 수출_억달러 DESC LIMIT 5
''')

## 5. 무엇을 파는가 — 품목 구성

In [ ]:
# 5.1 HS2 대분류 수출 비중: 2007 vs 2025 (2025 상위 12)
hs2 = q('''
    WITH s AS (SELECT f.yyyymm//100 yr, SUBSTR(f.hs10,1,2) hs2, SUM(f.exp_dlr) v
               FROM fact_trade f WHERE f.yyyymm//100 IN (2007,2025) GROUP BY 1,2),
    sh AS (SELECT yr,hs2,100.0*v/SUM(v) OVER (PARTITION BY yr) pct FROM s)
    SELECT hs2 AS HS2,
           ROUND(MAX(CASE WHEN yr=2007 THEN pct END),1) AS 비중_2007,
           ROUND(MAX(CASE WHEN yr=2025 THEN pct END),1) AS 비중_2025
    FROM sh GROUP BY hs2 ORDER BY 비중_2025 DESC NULLS LAST LIMIT 12
''')
print(hs2.to_string(index=False))

dd = hs2.dropna().head(10)
x = np.arange(len(dd)); w=0.4
plt.figure(figsize=(8,4))
plt.bar(x-w/2, dd['비중_2007'], w, label='2007', color='lightgray')
plt.bar(x+w/2, dd['비중_2025'], w, label='2025', color='steelblue')
plt.xticks(x, dd['HS2']); plt.ylabel('Share of exports (%)'); plt.xlabel('HS2 code')
plt.title('Export Composition by HS2 (2007 vs 2025)')
plt.legend(); plt.grid(alpha=.3, axis='y'); plt.tight_layout(); plt.show()

In [ ]:
# 5.2 2025년 수출 상위 10 품목 (HS10, 억달러)
q('''
    SELECT f.hs10 AS HS10, d.name_ko AS 품목명, ROUND(SUM(f.exp_dlr)/1e8,0) AS 수출_억달러
    FROM fact_trade f LEFT JOIN dim_hs10 d USING(hs10)
    WHERE f.yyyymm//100=2025 GROUP BY 1,2 ORDER BY 수출_억달러 DESC LIMIT 10
''')

## 6. 얼마나 집중돼 있는가 — 수출 90% 도달 HS6 수

In [ ]:
q('''
    WITH h AS (SELECT yyyymm//100 yr, SUBSTR(hs10,1,6) hs6, SUM(exp_dlr) v
               FROM fact_trade WHERE yyyymm//100 IN (2007,2010,2015,2020,2025) AND exp_dlr>0 GROUP BY 1,2),
    r AS (SELECT yr,hs6,v, SUM(v) OVER (PARTITION BY yr) tot,
                 SUM(v) OVER (PARTITION BY yr ORDER BY v DESC ROWS BETWEEN UNBOUNDED PRECEDING AND CURRENT ROW) cum
          FROM h)
    SELECT yr AS 연도,
           COUNT(*) FILTER (WHERE cum<=0.9*tot)+1 AS HS6수_상위90pct,
           COUNT(*) AS HS6수_전체
    FROM r GROUP BY yr ORDER BY yr
''')

## 7. 개정을 걸쳐 품목 잇기 — concordance 시연 ★

HS 코드는 개정 때 분할·통합·폐지된다. `dim_hs6_concordance`로 각 시기 HS6를 HS2022 안정코드로 환산한다. 다대다라 종수에 근사 오차가 있다(봉인 2: 완전 사각지대 종수 124/63/34, 거래액 0.004~0.007%).

In [ ]:
q('''
    WITH fh AS (SELECT yyyymm//100 yr, SUBSTR(hs10,1,6) hs6 FROM fact_trade
                WHERE yyyymm//100 IN (2007,2015,2025) AND exp_dlr>0 GROUP BY 1,2),
    ver AS (SELECT yr,hs6, CASE WHEN yr BETWEEN 2007 AND 2011 THEN '2007'
                                WHEN yr BETWEEN 2012 AND 2016 THEN '2012'
                                WHEN yr BETWEEN 2017 AND 2021 THEN '2017' ELSE '2022' END hv FROM fh),
    m AS (SELECT v.yr,v.hs6, CASE WHEN v.hv='2022' THEN v.hs6 ELSE cc.hs2022 END AS scode
          FROM ver v LEFT JOIN dim_hs6_concordance cc
               ON v.hv<>'2022' AND cc.past_version=v.hv AND cc.hs_past=v.hs6)
    SELECT yr AS 연도, COUNT(DISTINCT hs6) AS 원본HS6, COUNT(DISTINCT scode) AS 안정코드,
           COUNT(DISTINCT CASE WHEN scode IS NULL THEN hs6 END) AS 미매칭
    FROM m GROUP BY yr ORDER BY yr
''')

## 8. 얼마나 비싼가 — 단가(USD/kg) 분포

In [ ]:
q('''
    WITH u AS (SELECT SUBSTR(hs10,1,6) hs6, SUM(exp_dlr)*1.0/SUM(exp_wgt) up
               FROM fact_trade WHERE yyyymm//100=2025 AND exp_wgt>0 AND exp_dlr>0 GROUP BY 1)
    SELECT COUNT(*) AS HS6수,
           ROUND(quantile_cont(up,0.1),2) AS p10,
           ROUND(quantile_cont(up,0.5),2) AS 중앙값,
           ROUND(quantile_cont(up,0.9),2) AS p90
    FROM u
''')

## 9. 무역수지 — 흑자와 적자의 해

In [ ]:
plt.figure(figsize=(8,3.6))
colors = ['crimson' if b<0 else 'seagreen' for b in annual['bal_eok']]
plt.bar(annual['yr'], annual['bal_eok']/10, color=colors)
plt.axhline(0, color='black', lw=.8)
plt.ylabel('Billion USD'); plt.xlabel('Year')
plt.title('Korea Annual Trade Balance (2007-2025)')
plt.gca().xaxis.set_major_locator(MaxNLocator(integer=True))
plt.grid(alpha=.3, axis='y'); plt.tight_layout(); plt.show()

# 적자였던 해
print(annual[annual['bal_eok']<0].to_string(index=False))

## 마무리

모든 셀은 관측된 구조를 **서술**할 뿐 인과를 주장하지 않는다. 한계: 2026 부분년 제외, 음수·0 중량 제외, 개정 연결 근사 오차(봉인 2), `stat_cd` 집계코드 혼입 가능성, 국가 기준(수출=최종목적국·수입=원산국). 상세는 대시보드 "데이터 함정" 탭 참조.

In [ ]:
con.close()
print("연결 종료.")